# 05 — Joint-State Kalman Filter

**Repository:** `quantum-hardware-company`  
**Directory:** `control-stack/`

This notebook upgrades the control stack from independent parameter filters to a joint Ω/B state estimator.

## Purpose

Notebook 04 showed that adding velocity increased observability but did not automatically improve predictive control.

This notebook asks:

> Can joint covariance modeling improve estimation when calibration parameters are coupled?

## State model

Instead of filtering Ω drift and B drift independently:

```text
Ω filter
B filter
```

we estimate a joint state:

```text
x = [δΩ, δB]
```

and allow the process covariance to include coupling.

## Policies compared

- no compensation
- moving average
- independent scalar Kalman
- joint Kalman
- oracle baseline

## Principle

Quantum hardware parameters are often not independent.

A joint filter can represent shared drift structure and correlated uncertainty.


## Joint-state model

State:

$$
x_k =
\begin{bmatrix}
\delta\Omega_k \\
\delta B_k
\end{bmatrix}
$$

Transition:

$$
x_k = F x_{k-1} + w_k
$$

with:

$$
F =
\begin{bmatrix}
1 & 0 \\
0 & 1
\end{bmatrix}
$$

Measurement:

$$
z_k =
\begin{bmatrix}
\widehat{\delta\Omega}_k \\
\widehat{\delta B}_k
\end{bmatrix}
=
H x_k + v_k
$$

where:

$$
H = I
$$

The upgrade is in covariance:

$$
Q =
\begin{bmatrix}
q_\Omega & q_{\Omega B} \\
q_{\Omega B} & q_B
\end{bmatrix}
$$

A nonzero off-diagonal term lets the filter represent coupled drift.


In [ ]:
import os
import json

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(9423)

FIG_DIR = "figures/joint_state_kalman"
RESULTS_DIR = "results"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

def save_fig(name):
    """Save figure as PNG and PDF using the Notebook 05 naming convention."""
    plt.savefig(f"{FIG_DIR}/05_joint_state_kalman_{name}.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"{FIG_DIR}/05_joint_state_kalman_{name}.pdf", bbox_inches="tight")


In [ ]:
def rabi_model(t, A, Omega, B):
    return A * np.sin(0.5 * Omega * t) ** 2 + B


def fit_model(model, x, y, p0, bounds=(-np.inf, np.inf)):
    params, cov = curve_fit(model, x, y, p0=p0, bounds=bounds, maxfev=30000)
    return params, cov


def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


def phase_lock_metric(signal, target):
    dot = np.dot(signal, target)
    norm = np.linalg.norm(signal) * np.linalg.norm(target)
    if norm == 0:
        return np.nan
    return dot / norm


def moving_average(x, window=4):
    y = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        lo = max(0, i - window + 1)
        y[i] = np.mean(x[lo:i + 1])
    return y


def kalman_filter_1d(z, q, r, x0=None, p0=1.0):
    z = np.asarray(z, dtype=float)
    x_hat = np.zeros_like(z)
    p_hist = np.zeros_like(z)
    k_hist = np.zeros_like(z)

    x = z[0] if x0 is None else float(x0)
    p = float(p0)

    for i, measurement in enumerate(z):
        x_pred = x
        p_pred = p + q

        k_gain = p_pred / (p_pred + r)
        x = x_pred + k_gain * (measurement - x_pred)
        p = (1 - k_gain) * p_pred

        x_hat[i] = x
        p_hist[i] = p
        k_hist[i] = k_gain

    return x_hat, p_hist, k_hist


def kalman_filter_joint(z, F, H, Q, R, x0=None, P0=None):
    """Linear Kalman filter for a joint state.

    z: array with shape (n_steps, n_measurements)
    F: transition matrix
    H: measurement matrix
    Q: process covariance
    R: measurement covariance
    """
    z = np.asarray(z, dtype=float)
    n_steps = z.shape[0]
    n_state = F.shape[0]

    if x0 is None:
        x = z[0].reshape(n_state, 1)
    else:
        x = np.asarray(x0, dtype=float).reshape(n_state, 1)

    if P0 is None:
        P = np.eye(n_state)
    else:
        P = np.asarray(P0, dtype=float)

    x_filt = np.zeros((n_steps, n_state))
    x_pred = np.zeros((n_steps, n_state))
    P_trace = np.zeros((n_steps, n_state, n_state))
    K_trace = np.zeros((n_steps, n_state, H.shape[0]))

    I = np.eye(n_state)

    for i in range(n_steps):
        # Predict
        x_prior = F @ x
        P_prior = F @ P @ F.T + Q

        # Update
        y = z[i].reshape(-1, 1) - H @ x_prior
        S = H @ P_prior @ H.T + R
        K = P_prior @ H.T @ np.linalg.inv(S)

        x = x_prior + K @ y
        P = (I - K @ H) @ P_prior

        x_pred[i] = x_prior.ravel()
        x_filt[i] = x.ravel()
        P_trace[i] = P
        K_trace[i] = K

    return x_filt, x_pred, P_trace, K_trace


def evaluate_policy(name, delta_omega_hat, delta_b_hat, true_delta_omega, true_delta_b):
    omega_controlled = target_Omega + true_delta_omega - delta_omega_hat
    b_controlled = np.clip(target_B + true_delta_b - delta_b_hat, 0, 0.3)

    target_signal_local = rabi_model(pulse_t, target_A, target_Omega, target_B)

    response_rmse = []
    phase_lock = []
    controlled_signals = []

    for k in range(len(true_delta_omega)):
        y = rabi_model(pulse_t, target_A, omega_controlled[k], b_controlled[k])
        controlled_signals.append(y)
        response_rmse.append(rmse(y, target_signal_local))
        phase_lock.append(phase_lock_metric(y, target_signal_local))

    response_rmse = np.array(response_rmse)
    phase_lock = np.array(phase_lock)
    controlled_signals = np.array(controlled_signals)

    return {
        "name": name,
        "omega_controlled": omega_controlled,
        "b_controlled": b_controlled,
        "omega_error": omega_controlled - target_Omega,
        "b_error": b_controlled - target_B,
        "omega_rmse": rmse(omega_controlled, np.full(len(omega_controlled), target_Omega)),
        "b_rmse": rmse(b_controlled, np.full(len(b_controlled), target_B)),
        "response_rmse": response_rmse,
        "response_rmse_mean": float(np.mean(response_rmse)),
        "response_rmse_max": float(np.max(response_rmse)),
        "phase_lock": phase_lock,
        "min_phase_lock": float(np.min(phase_lock)),
        "mean_phase_lock": float(np.mean(phase_lock)),
        "max_angle_degrees": float(np.max(np.degrees(np.arccos(np.clip(phase_lock, -1, 1))))),
        "controlled_signals": controlled_signals,
    }


## Generate coupled synthetic drift environment

Notebook 05 introduces weak coupling between Ω and B drift.

This makes joint-state filtering meaningful: the two parameters share part of the same latent drift structure.


In [ ]:
n_blocks = 48
block_times = np.arange(n_blocks)

n_points_per_block = 140
pulse_t = np.linspace(0, 10, n_points_per_block)

target_A = 0.88
target_Omega = 2.45
target_B = 0.06

target_signal = rabi_model(pulse_t, target_A, target_Omega, target_B)

# Shared latent drift component
shared = 0.018 * np.sin(2 * np.pi * block_times / 23 + 0.25)

# Parameter-specific drift
omega_private = (
    0.050 * np.sin(2 * np.pi * block_times / 21 + 0.4)
    + 0.014 * np.sin(2 * np.pi * block_times / 9)
)

b_private = (
    0.0013 * block_times
    + 0.005 * np.sin(2 * np.pi * block_times / 14 + 0.2)
)

# Coupled true drifts
true_delta_Omega = omega_private + shared
true_delta_B = b_private + 0.22 * shared

Omega_uncomp = target_Omega + true_delta_Omega
B_uncomp = target_B + true_delta_B
A_uncomp = target_A + 0.012 * np.sin(2 * np.pi * block_times / 17)

print(f"n_blocks = {n_blocks}")
print(f"true Ω/B drift correlation = {np.corrcoef(true_delta_Omega, true_delta_B)[0,1]:.3f}")


## Simulate calibration measurements and fit each block


In [ ]:
Y_obs = []
Y_clean = []

for k in range(n_blocks):
    y_clean = rabi_model(pulse_t, A_uncomp[k], Omega_uncomp[k], B_uncomp[k])
    noise = 0.04 * np.random.randn(n_points_per_block)
    y_obs = np.clip(y_clean + noise, 0, 1)
    Y_clean.append(y_clean)
    Y_obs.append(y_obs)

Y_clean = np.array(Y_clean)
Y_obs = np.array(Y_obs)

fit_params = []
fit_stds = []

initial_guess = [0.85, 2.40, 0.05]
bounds = ([0.0, 0.0, 0.0], [1.2, 5.0, 0.3])

for k in range(n_blocks):
    params, cov = fit_model(
        rabi_model,
        pulse_t,
        Y_obs[k],
        p0=initial_guess,
        bounds=bounds,
    )
    fit_params.append(params)
    fit_stds.append(np.sqrt(np.diag(cov)))

fit_params = np.array(fit_params)
fit_stds = np.array(fit_stds)

Omega_est = fit_params[:, 1]
B_est = fit_params[:, 2]

delta_Omega_est = Omega_est - target_Omega
delta_B_est = B_est - target_B

print("Calibration fit complete.")
print(f"measured Ω/B drift correlation = {np.corrcoef(delta_Omega_est, delta_B_est)[0,1]:.3f}")


## Define independent and joint estimators


In [ ]:
# No compensation
delta_Omega_none = np.zeros(n_blocks)
delta_B_none = np.zeros(n_blocks)

# Moving average baseline
ma_window = 4
delta_Omega_ma = moving_average(delta_Omega_est, window=ma_window)
delta_B_ma = moving_average(delta_B_est, window=ma_window)

# Independent scalar Kalman filters
r_omega = float(np.median(fit_stds[:, 1] ** 2))
r_b = float(np.median(fit_stds[:, 2] ** 2))

q_omega = 8.0e-4
q_b = 1.0e-5

delta_Omega_scalar, _, _ = kalman_filter_1d(delta_Omega_est, q=q_omega, r=r_omega, p0=1.0)
delta_B_scalar, _, _ = kalman_filter_1d(delta_B_est, q=q_b, r=r_b, p0=1.0)

# Joint Kalman filter
z_joint = np.column_stack([delta_Omega_est, delta_B_est])

F = np.eye(2)
H = np.eye(2)

# Measurement covariance from fit uncertainty, with small measurement cross-covariance estimate.
R = np.array([
    [r_omega, 0.0],
    [0.0, r_b],
])

# Coupled process covariance.
q_cross = 0.20 * np.sqrt(q_omega * q_b)
Q_joint = np.array([
    [q_omega, q_cross],
    [q_cross, q_b],
])

joint_filt, joint_pred, P_joint, K_joint = kalman_filter_joint(
    z_joint,
    F=F,
    H=H,
    Q=Q_joint,
    R=R,
    P0=np.eye(2),
)

delta_Omega_joint = joint_filt[:, 0]
delta_B_joint = joint_filt[:, 1]

# Oracle
delta_Omega_oracle = true_delta_Omega.copy()
delta_B_oracle = true_delta_B.copy()

print("Estimators ready.")
print("Q_joint:")
print(Q_joint)
print("R:")
print(R)


## Estimator comparison

The joint filter is allowed to use Ω/B covariance while independent filters cannot.


In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(block_times, true_delta_Omega, linewidth=2, label="true Ω drift")
plt.plot(block_times, delta_Omega_est, marker="o", linewidth=1, label="measured Ω drift")
plt.plot(block_times, delta_Omega_ma, linewidth=2, linestyle="--", label="moving average")
plt.plot(block_times, delta_Omega_scalar, linewidth=2, linestyle=":", label="independent scalar Kalman")
plt.plot(block_times, delta_Omega_joint, linewidth=2, linestyle="-.", label="joint Kalman")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("Ω drift estimate")
plt.title("Joint-state estimator comparison: Rabi frequency")
plt.legend()
plt.tight_layout()

save_fig("omega_estimator_comparison")

plt.show()


In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(block_times, true_delta_B, linewidth=2, label="true B drift")
plt.plot(block_times, delta_B_est, marker="o", linewidth=1, label="measured B drift")
plt.plot(block_times, delta_B_ma, linewidth=2, linestyle="--", label="moving average")
plt.plot(block_times, delta_B_scalar, linewidth=2, linestyle=":", label="independent scalar Kalman")
plt.plot(block_times, delta_B_joint, linewidth=2, linestyle="-.", label="joint Kalman")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("B drift estimate")
plt.title("Joint-state estimator comparison: readout offset")
plt.legend()
plt.tight_layout()

save_fig("offset_estimator_comparison")

plt.show()


## Covariance diagnostics

Track joint posterior covariance to see whether the filter carries Ω/B coupling information.


In [ ]:
posterior_cov_cross = P_joint[:, 0, 1]
posterior_corr = P_joint[:, 0, 1] / np.sqrt(P_joint[:, 0, 0] * P_joint[:, 1, 1])

plt.figure(figsize=(10, 5))
plt.plot(block_times, posterior_cov_cross, marker="o", linewidth=1, label="posterior covariance Ω/B")
plt.plot(block_times, posterior_corr, marker="o", linewidth=1, label="posterior correlation Ω/B")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("covariance / correlation")
plt.title("Joint Kalman covariance diagnostics")
plt.legend()
plt.tight_layout()

save_fig("covariance_diagnostics")

plt.show()


## Evaluate policies


In [ ]:
policies = {
    "none": (delta_Omega_none, delta_B_none),
    "moving_average": (delta_Omega_ma, delta_B_ma),
    "independent_scalar": (delta_Omega_scalar, delta_B_scalar),
    "joint_kalman": (delta_Omega_joint, delta_B_joint),
    "oracle": (delta_Omega_oracle, delta_B_oracle),
}

policy_results = {}

for name, (delta_omega_hat, delta_b_hat) in policies.items():
    policy_results[name] = evaluate_policy(
        name,
        delta_omega_hat,
        delta_b_hat,
        true_delta_Omega,
        true_delta_B,
    )

for name, result in policy_results.items():
    print(
        f"{name:20s} | "
        f"response RMSE mean = {result['response_rmse_mean']:.6f} | "
        f"max = {result['response_rmse_max']:.6f} | "
        f"min phase-lock = {result['min_phase_lock']:.6f}"
    )


## Response-level error comparison


In [ ]:
plt.figure(figsize=(11, 5))
for name, result in policy_results.items():
    plt.plot(block_times, result["response_rmse"], marker="o", linewidth=1, label=name)

plt.xlabel("calibration block / lab time index")
plt.ylabel("RMSE vs target response")
plt.title("Joint-state control: response-level error comparison")
plt.legend()
plt.tight_layout()

save_fig("response_rmse_comparison")

plt.show()


## Parameter error comparison


In [ ]:
plt.figure(figsize=(11, 5))
for name, result in policy_results.items():
    if name == "oracle":
        continue
    plt.plot(block_times, result["omega_error"], marker="o", linewidth=1, label=name)

plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("Ω error from target")
plt.title("Joint-state control: Ω error comparison")
plt.legend()
plt.tight_layout()

save_fig("omega_error_comparison")

plt.show()


In [ ]:
plt.figure(figsize=(11, 5))
for name, result in policy_results.items():
    if name == "oracle":
        continue
    plt.plot(block_times, result["b_error"], marker="o", linewidth=1, label=name)

plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("B error from target")
plt.title("Joint-state control: B error comparison")
plt.legend()
plt.tight_layout()

save_fig("offset_error_comparison")

plt.show()


## CGCS phase-lock stability


In [ ]:
threshold = 1 / np.sqrt(2)

plt.figure(figsize=(11, 5))
plt.axhline(threshold, linewidth=1, linestyle="--", label="45° threshold")

for name, result in policy_results.items():
    plt.plot(block_times, result["phase_lock"], marker="o", linewidth=1, label=name)

plt.ylim(0.68, 1.01)
plt.xlabel("calibration block / lab time index")
plt.ylabel("cosine similarity vs target")
plt.title("Joint-state control: CGCS phase-lock stability")
plt.legend()
plt.tight_layout()

save_fig("cgcs_stability_comparison")

plt.show()


## Worst-case block comparison


In [ ]:
worst_block = int(np.argmax(policy_results["none"]["response_rmse"]))

plt.figure(figsize=(11, 5))
plt.plot(pulse_t, target_signal, linewidth=3, label="target")

for name, result in policy_results.items():
    plt.plot(pulse_t, result["controlled_signals"][worst_block], linewidth=2, label=name)

plt.xlabel("time / pulse duration")
plt.ylabel("excited-state probability")
plt.title(f"Joint-state control: worst-case block {worst_block}")
plt.legend()
plt.tight_layout()

save_fig("worst_case_block_comparison")

plt.show()


## Policy ranking


In [ ]:
ranking = sorted(policy_results.values(), key=lambda r: r["response_rmse_mean"])

policy_table = []
for rank, result in enumerate(ranking, start=1):
    policy_table.append({
        "rank": rank,
        "policy": result["name"],
        "omega_rmse": result["omega_rmse"],
        "b_rmse": result["b_rmse"],
        "response_rmse_mean": result["response_rmse_mean"],
        "response_rmse_max": result["response_rmse_max"],
        "min_phase_lock": result["min_phase_lock"],
        "mean_phase_lock": result["mean_phase_lock"],
        "max_angle_degrees": result["max_angle_degrees"],
        "all_blocks_phase_locked": bool(np.all(result["phase_lock"] >= threshold)),
    })

policy_table


In [ ]:
policy_names_ranked = [row["policy"] for row in policy_table]
response_means_ranked = [row["response_rmse_mean"] for row in policy_table]
response_max_ranked = [row["response_rmse_max"] for row in policy_table]

x = np.arange(len(policy_names_ranked))
width = 0.35

plt.figure(figsize=(11, 5))
bars_mean = plt.bar(x - width / 2, response_means_ranked, width, label="mean RMSE")
bars_max = plt.bar(x + width / 2, response_max_ranked, width, label="max RMSE")

plt.xticks(x, policy_names_ranked, rotation=25, ha="right")
plt.ylabel("response RMSE")
plt.title("Joint-state control: policy ranking")
plt.legend()

for bars in [bars_mean, bars_max]:
    for bar in bars:
        value = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.002,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )

plt.tight_layout()

save_fig("policy_ranking_summary")

plt.show()


## Coupling sweep

Sweep the process covariance coupling term to test whether shared drift improves or degrades response-level control.


In [ ]:
coupling_values = np.linspace(-0.8, 0.8, 33)
sweep_response_rmse = []
sweep_joint_corr = []

for c in coupling_values:
    q_cross_test = c * np.sqrt(q_omega * q_b)
    Q_test = np.array([
        [q_omega, q_cross_test],
        [q_cross_test, q_b],
    ])

    try:
        joint_test, _, _, _ = kalman_filter_joint(
            z_joint,
            F=F,
            H=H,
            Q=Q_test,
            R=R,
            P0=np.eye(2),
        )
        test_eval = evaluate_policy(
            "joint_test",
            joint_test[:, 0],
            joint_test[:, 1],
            true_delta_Omega,
            true_delta_B,
        )
        sweep_response_rmse.append(test_eval["response_rmse_mean"])
        sweep_joint_corr.append(c)
    except np.linalg.LinAlgError:
        sweep_response_rmse.append(np.nan)
        sweep_joint_corr.append(c)

sweep_response_rmse = np.array(sweep_response_rmse)
best_idx = int(np.nanargmin(sweep_response_rmse))
best_coupling = float(coupling_values[best_idx])

print(f"Current coupling = 0.20")
print(f"Best swept coupling = {best_coupling:.3f}")
print(f"Best swept response RMSE = {sweep_response_rmse[best_idx]:.6f}")


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(coupling_values, sweep_response_rmse, marker="o", linewidth=1)
plt.axvline(0.20, linewidth=1, linestyle="--", label="current coupling = 0.20")
plt.axvline(best_coupling, linewidth=1, linestyle=":", label=f"best coupling = {best_coupling:.2f}")
plt.xlabel("process covariance coupling coefficient")
plt.ylabel("mean response RMSE")
plt.title("Joint-state Kalman coupling sweep")
plt.legend()
plt.tight_layout()

save_fig("coupling_sweep")

plt.show()


## Save joint-state summary


In [ ]:
summary = {
    "n_blocks": int(n_blocks),
    "moving_average_window": int(ma_window),
    "q_omega": float(q_omega),
    "q_b": float(q_b),
    "q_cross": float(q_cross),
    "coupling_coefficient": float(0.20),
    "best_swept_coupling": float(best_coupling),
    "best_swept_coupling_response_rmse": float(sweep_response_rmse[best_idx]),
    "r_omega": float(r_omega),
    "r_b": float(r_b),
    "true_drift_correlation": float(np.corrcoef(true_delta_Omega, true_delta_B)[0,1]),
    "measured_drift_correlation": float(np.corrcoef(delta_Omega_est, delta_B_est)[0,1]),
    "phase_lock_threshold": float(threshold),
    "ranking": policy_table,
}

with open(f"{RESULTS_DIR}/joint_state_kalman_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved {RESULTS_DIR}/joint_state_kalman_summary.json")


## Zip outputs for download from Colab


In [ ]:
!zip -r joint_state_kalman_outputs.zip figures results


## Control-stack status

This notebook tests whether covariance-aware joint filtering improves drift control.

## Interpretation

Joint-state filtering matters when Ω and B drift share structure.

If the coupling sweep favors nonzero coupling, the hardware stack is telling us that calibration parameters are not independent.

## Next steps

Possible next notebooks:

- `06_model_predictive_control_lite.ipynb` — use horizon-based predicted drift.
- `noise-mitigation/01_structured_drift_model.ipynb` — model residuals after control.
- `control-stack/docs/` — write joint-state technical summary.

Guiding rule:

> Shared drift should be modeled as shared state, not independent noise.
